# 04 — Results aggregation: controlled + best-per-stock tables with multi-seed CIs

Reads `results/raw/*.csv` and `results/selected_config.json`, then produces:
- **Primary headline table** — controlled (model=LightGBM, h=5), all stocks, all ablations, mean±std over 5 seeds.
- **Secondary table** — best-per-stock by val (locked in `selected_config.json`), reporting DSR.
- **Bootstrap CIs** on Sharpe and annual return using stationary block bootstrap.
- LaTeX-ready output snippets.

In [1]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as cfg_mod
from src import metrics as M
from src.latex_utils import to_booktabs_latex

cfg = cfg_mod.load_config()
RAW = cfg_mod.results_dir() / 'raw'
df = pd.concat([pd.read_csv(p) for p in sorted(RAW.glob('*.csv'))], ignore_index=True)
print(f'Loaded {len(df):,} rows across {df["Model"].nunique()} models')
df.head()

Loaded 4,200 rows across 7 models


,Stock,Ablation,Model,Horizon,Threshold,Seed,Best_Params,Val_Accuracy,Val_ROC_AUC,Val_Trades,...,Test_Accuracy,Test_ROC_AUC,Test_Trades,Test_WinRate,Test_Sharpe,Test_AnnualReturn,Test_MDD,Test_Sortino,Test_Calmar,HParams
0,AAPL,full_70_30,gradient_boosting,2,0.0025,42,"{""n_estimators"": ""328"", ""learning_rate"": 0.080...",0.536290,0.528148,123,...,0.512195,0.516104,122,0.459016,-0.300556,-0.174636,-0.260975,-0.520040,-0.669167,NaN
1,AAPL,full_70_30,gradient_boosting,2,0.0025,7,"{""n_estimators"": ""50"", ""learning_rate"": 0.2880...",0.524194,0.536671,123,...,0.516260,0.517593,122,0.459016,0.566015,0.234325,-0.148246,1.097500,1.580647,NaN
2,AAPL,full_70_30,gradient_boosting,2,0.0025,1337,"{""n_estimators"": ""234"", ""learning_rate"": 0.068...",0.520161,0.528540,123,...,0.520325,0.522520,122,0.434426,-0.095895,-0.091879,-0.232826,-0.187045,-0.394627,NaN
3,AAPL,full_70_30,gradient_boosting,2,0.0025,2024,"{""n_estimators"": ""377"", ""learning_rate"": 0.063...",0.528226,0.534744,123,...,0.520325,0.524008,122,0.442623,-0.375050,-0.202786,-0.279710,-0.650830,-0.724987,NaN
4,AAPL,full_70_30,gradient_boosting,2,0.0025,31415,"{""n_estimators"": ""50"", ""learning_rate"": 0.1499...",0.479839,0.525470,123,...,0.471545,0.487897,122,0.434426,-0.650780,-0.298929,-0.350998,-1.094970,-0.851653,NaN


In [2]:
# Primary headline table: controlled model + horizon, all stocks, all ablations, mean over seeds.
ctrl_model = cfg['controlled']['model']
ctrl_h = cfg['controlled']['horizon']
sub = df[(df['Model'] == ctrl_model) & (df['Horizon'] == ctrl_h)].copy()
agg = (sub.groupby(['Stock', 'Ablation'])
         .agg(Test_AUC=('Test_ROC_AUC', 'mean'),
              Test_AUC_std=('Test_ROC_AUC', 'std'),
              Test_Sharpe=('Test_Sharpe', 'mean'),
              Test_Sharpe_std=('Test_Sharpe', 'std'),
              Test_AnnualReturn=('Test_AnnualReturn', 'mean'),
              Test_MDD=('Test_MDD', 'mean'),
              Test_WinRate=('Test_WinRate', 'mean'),
              n_seeds=('Seed', 'count'))
         .reset_index())
agg.round(3)

,Stock,Ablation,Test_AUC,Test_AUC_std,Test_Sharpe,Test_Sharpe_std,Test_AnnualReturn,Test_MDD,Test_WinRate,n_seeds
0,AAPL,equal_50_50,0.499,0.003,-0.952,0.195,-0.754,-0.329,0.376,5
1,AAPL,full_70_30,0.474,0.015,-1.640,0.069,-0.900,-0.453,0.416,5
2,AAPL,learned_alpha_global,0.511,0.009,-1.231,0.231,-0.827,-0.335,0.400,5
3,AAPL,learned_alpha_per_stock,0.550,0.001,-0.754,0.096,-0.688,-0.268,0.404,5
4,AAPL,news_only,0.525,0.010,-1.606,0.068,-0.895,-0.365,0.327,5
5,AAPL,sentiment_only,0.538,0.001,-1.275,0.000,-0.842,-0.378,0.408,5
6,AAPL,social_only,0.471,0.010,-2.196,0.318,-0.946,-0.512,0.306,5
7,AAPL,technical_only,0.543,0.019,0.561,0.330,0.906,-0.260,0.527,5
8,META,equal_50_50,0.507,0.017,0.636,0.201,1.449,-0.264,0.490,5
9,META,full_70_30,0.469,0.007,-1.074,1.040,-0.672,-0.432,0.424,5


In [3]:
# Secondary: best-per-stock under each criterion, with DSR adjustment over the number of trials.
sel_path = cfg_mod.selected_config_path()
if sel_path.exists():
    sel = json.loads(sel_path.read_text())
    print(json.dumps(sel, indent=2)[:1200], '...')
else:
    print('selected_config.json not yet present — run src.select_best_config first.')

{
  "AAPL": {
    "equal_50_50": {
      "val_auc": {
        "Model": "logistic_regression",
        "Horizon": 2,
        "mean_val": 0.6051593521421108,
        "std_val": 2.920673951802145e-05,
        "n_seeds": 5
      },
      "val_sharpe": {
        "Model": "logistic_regression",
        "Horizon": 10,
        "mean_val": 2.146597188089258,
        "std_val": 0.0,
        "n_seeds": 5
      }
    },
    "full_70_30": {
      "val_auc": {
        "Model": "logistic_regression",
        "Horizon": 2,
        "mean_val": 0.6324973876698015,
        "std_val": 0.0003677034550746799,
        "n_seeds": 5
      },
      "val_sharpe": {
        "Model": "logistic_regression",
        "Horizon": 10,
        "mean_val": 2.3509501942379143,
        "std_val": 0.0,
        "n_seeds": 5
      }
    },
    "learned_alpha_global": {
      "val_auc": {
        "Model": "logistic_regression",
        "Horizon": 2,
        "mean_val": 0.6379049111807732,
        "std_val": 0.000182396029829871

In [4]:
# DSR helper requires trade-level returns. Save them in your HPO runs (the `__test_trade_returns__` field
# in hpo_traditional.py.evaluate_cell). For now, demonstrate the DSR call on synthetic data.
from scipy import stats as scs
rng = np.random.default_rng(0)
r = rng.normal(0.002, 0.02, 60)
obs_sharpe = M.sharpe_ratio(r)
skew = scs.skew(r); kurt = scs.kurtosis(r, fisher=True)
dsr = M.deflated_sharpe_ratio(obs_sharpe, n_trials=100, n_trades=len(r),
                              skew_trade_returns=skew,
                              excess_kurtosis_trade_returns=kurt)
print(f'observed Sharpe={obs_sharpe:.3f}, DSR over 100 trials = {dsr:.3f}')


observed Sharpe=1.530, DSR over 100 trials = 0.000


In [5]:
# Bootstrap CI helper demo (on the same synthetic returns).
pt, lo, hi = M.bootstrap_metric_ci(r, M.sharpe_ratio, n_boot=1000, seed=0)
print(f'Sharpe = {pt:.3f}  95% CI [{lo:.3f}, {hi:.3f}]')

Sharpe = 1.530  95% CI [-1.032, 4.129]


In [6]:
# Export the controlled table as a LaTeX-ready snippet.
tbl = agg.copy()
for col in ['Test_AUC', 'Test_Sharpe', 'Test_AnnualReturn', 'Test_MDD', 'Test_WinRate']:
    tbl[col] = tbl[col].round(3)
print(to_booktabs_latex(tbl))

\begin{tabular}{llllllllll}
\toprule
Stock & Ablation & Test\_AUC & Test\_AUC\_std & Test\_Sharpe & Test\_Sharpe\_std & Test\_AnnualReturn & Test\_MDD & Test\_WinRate & n\_seeds \\
\midrule
AAPL & equal\_50\_50 & 0.4990 & 0.0028 & -0.9520 & 0.1952 & -0.7540 & -0.3290 & 0.3760 & 5 \\
AAPL & full\_70\_30 & 0.4740 & 0.0146 & -1.6400 & 0.0685 & -0.9000 & -0.4530 & 0.4160 & 5 \\
AAPL & learned\_alpha\_global & 0.5110 & 0.0086 & -1.2310 & 0.2313 & -0.8270 & -0.3350 & 0.4000 & 5 \\
AAPL & learned\_alpha\_per\_stock & 0.5500 & 0.0008 & -0.7540 & 0.0964 & -0.6880 & -0.2680 & 0.4040 & 5 \\
AAPL & news\_only & 0.5250 & 0.0101 & -1.6060 & 0.0685 & -0.8950 & -0.3650 & 0.3270 & 5 \\
AAPL & sentiment\_only & 0.5380 & 0.0014 & -1.2750 & 0.0000 & -0.8420 & -0.3780 & 0.4080 & 5 \\
AAPL & social\_only & 0.4710 & 0.0102 & -2.1960 & 0.3179 & -0.9460 & -0.5120 & 0.3060 & 5 \\
AAPL & technical\_only & 0.5430 & 0.0185 & 0.5610 & 0.3297 & 0.9060 & -0.2600 & 0.5270 & 5 \\
META & equal\_50\_50 & 0.5070 & 0.0166 